## **SNAP Jupyter demo notebook**
# GSLC InSAR vs Classical InSAR — Mount Etna (Sentinel-1 IW)

##### ***About the test data:***
A 12-day Mount Etna Sentinel-1 IW SLC pair (2014-08-09 / 2014-08-21, same track, IW1, VV). The
notebook runs **two** interferometric pipelines on the *same* burst and geocodes both to map geometry:

* **Classical** — Back-Geocoding (DEM-assisted) coregistration → Interferogram → TOPSAR-Deburst →
  TopoPhaseRemoval → Goldstein → Terrain-Correction.
* **GSLC** — geocode the master with `GSLC-Terrain-Correction`, then `CreateStack` (auto
  cross-correlation bias) the raw slave → Interferogram → Goldstein (already geocoded).

It then compares coherence (histograms + statistics) and wrapped-phase / coherence maps side by side,
quantifying how GSLC InSAR compares to the classical workflow for Sentinel-1 TOPS.

##### ***Some information on the Python environment:***

In [ ]:
import os
# Make the gpt subprocess emit UTF-8 console text (snapista decodes gpt stdout as UTF-8; on Windows
# gpt would otherwise emit cp1252 bytes and snapista raises UnicodeDecodeError).
os.environ.setdefault('JAVA_TOOL_OPTIONS', '-Dsun.stdout.encoding=UTF-8 -Dsun.stderr.encoding=UTF-8')
import sys; print(sys.version)

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO
import snapista
from snapista import Graph
from snapista import Operator
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience functions:***

In [ ]:
def _read_dec(band, target=1400):
    """Read a band downsampled to ~target px on the long side, in row strips (avoids the
    'Cannot construct DataBuffer' that a full-resolution read of a large/virtual band triggers)."""
    w, h = band.getRasterWidth(), band.getRasterHeight()
    dec = max(1, int(max(w, h) / target)); cols = list(range(0, w, dec)); block = dec * 64; rows = []
    for y in range(0, h, block):
        bh = min(block, h - y)
        buf = np.zeros(w * bh, np.float32); band.readPixels(0, y, w, bh, buf)
        buf.shape = (bh, w); rows.append(buf[::dec][:, cols])
    return np.vstack(rows)

def _find(names, *prefixes):
    for p in prefixes:
        for n in names:
            if n.lower().startswith(p): return n
    return None

def _phase_coh(product):
    """Return (masked wrapped phase, masked coherence-or-None) from a geocoded interferogram product.
    Mask = imaged area only: pixels where the interferogram real & imag are both zero are the
    geocoded nodata border and are excluded."""
    names = [b.getName() for b in product.getBands()]
    iN = next(n for n in names if n.lower().startswith('i_ifg')
              or (n.lower().startswith('i_') and product.getBand(n).getUnit() == 'real'))
    qN = ('q_' + iN[2:]) if iN.lower().startswith('i_ifg') else iN.replace('i_', 'q_', 1)
    i = _read_dec(product.getBand(iN)); q = _read_dec(product.getBand(qN))
    mask = (i == 0) & (q == 0)
    phi = np.ma.masked_where(mask, np.arctan2(q, i))
    cohN = _find(names, 'coh')
    coh = np.ma.masked_where(mask, _read_dec(product.getBand(cohN))) if cohN else None
    return phi, coh

def coherence_samples(product):
    """1-D valid coherence samples (imaged area only)."""
    _, coh = _phase_coh(product)
    return coh.compressed() if coh is not None else np.array([])

def plot_phase_panels(pa, pb, ta, tb):
    fig, axs = plt.subplots(1, 2, figsize=(13, 6))
    for ax, p, t in ((axs[0], pa, ta), (axs[1], pb, tb)):
        phi, _ = _phase_coh(p)
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title(t + ' — wrapped phase [rad]'); fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()

def plot_coherence_panels(pa, pb, ta, tb):
    fig, axs = plt.subplots(1, 2, figsize=(13, 6))
    for ax, p, t in ((axs[0], pa, ta), (axs[1], pb, tb)):
        _, coh = _phase_coh(p)
        im = ax.imshow(coh, cmap='viridis', vmin=0, vmax=1)
        ax.set_title(t + ' — coherence'); fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()

def plot_coherence_hist(pa, pb, ta, tb):
    a, b = coherence_samples(pa), coherence_samples(pb)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(a, bins=50, range=(0, 1), density=True, alpha=0.5, label=ta)
    ax.hist(b, bins=50, range=(0, 1), density=True, alpha=0.5, label=tb)
    ax.set_xlabel('coherence'); ax.set_ylabel('density'); ax.legend()
    ax.set_title('Coherence distribution (imaged area)')
    plt.show()
    for name, v in ((ta, a), (tb, b)):
        if v.size:
            print('%-12s mean=%.3f  median=%.3f  N=%d' % (name, v.mean(), np.median(v), v.size))
        else:
            print('%-12s (no valid coherence samples)' % name)

---
##### ***Configure input paths and processing parameters***

In [ ]:
import urllib.request as _urlreq, zipfile as _zip, glob as _glob, subprocess
data_dir = os.path.join(os.getcwd(), 'data'); graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
for d in (data_dir, graphs_dir, results_dir): os.makedirs(d, exist_ok=True)

def fetch_cached(url, dest_dir):
    """Download url into dest_dir unless present, unzip a .zip, return manifest.safe (or the file).
    Cached: if the product is already extracted it is NOT downloaded again."""
    os.makedirs(dest_dir, exist_ok=True)
    fname = url.split('/')[-1]; stem = fname[:-4] if fname.lower().endswith('.zip') else fname
    hits = _glob.glob(os.path.join(dest_dir, stem + '*', 'manifest.safe'))
    if hits: print('cached:', os.path.basename(os.path.dirname(hits[0]))); return hits[0]
    local = os.path.join(dest_dir, fname)
    if not os.path.exists(local):
        print('downloading', fname, '...'); _urlreq.urlretrieve(url, local)
        print('  saved %.0f MB' % (os.path.getsize(local) / 1e6))
    if fname.lower().endswith('.zip'):
        with _zip.ZipFile(local) as z: z.extractall(dest_dir)
        hits = _glob.glob(os.path.join(dest_dir, stem + '*', 'manifest.safe'))
        return hits[0] if hits else local
    return local

def run_graph(g, xml_path):
    """Run a snapista Graph via gpt on its saved XML, console -> .log (avoids a Windows snapista
    pipe deadlock where gpt finishes but a child keeps stdout open)."""
    g.save_graph(xml_path); log_path = os.path.splitext(xml_path)[0] + '.log'
    with open(log_path, 'w', encoding='utf-8', errors='replace') as fh:
        rc = subprocess.run(['gpt', xml_path], stdout=fh, stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL).returncode
    if rc != 0:
        tail = open(log_path, encoding='utf-8', errors='replace').read()[-2000:]
        raise RuntimeError('gpt failed (rc=%d) on %s:\n%s' % (rc, os.path.basename(xml_path), tail))
    print('graph OK:', os.path.basename(xml_path)); return log_path

# Mount Etna IW SLC pair (12-day) on public S3; downloaded on first run (~4 GB each), cached in data/.
_etna = 'https://skywatch-public.s3.us-west-2.amazonaws.com/snap/data/S1/SLC/Etna-DLR/'
reference_slc = fetch_cached(_etna + 'S1A_IW_SLC__1SDV_20140809T165546_20140809T165613_001866_001C20_088B.zip', data_dir)
secondary_slc = fetch_cached(_etna + 'S1A_IW_SLC__1SDV_20140821T165547_20140821T165614_002041_001FC1_8601.zip', data_dir)

# Same extent / DEM / coherence window for both pipelines so the comparison is fair.
# Mt Etna sits in IW2 for this pair; wktAoi selects the bursts over the volcano (the raw IW1
# burst-2 used previously was over sea, lat ~36.5, which decorrelates).
SUBSWATH = 'IW2'; POL = 'VV'   # Mt Etna (37.75N) is in IW2 for this pair
ETNA_WKT = 'POLYGON((14.93 37.69,15.12 37.69,15.12 37.82,14.93 37.82,14.93 37.69))'
# NOTE: this runs full-width IW2 (~2 bursts). It is compute-heavy (~>1 h headless). A range-only
# Subset would speed up the GSLC branch, but TOPSAR-Deburst in the classical branch rejects a
# range-cropped split ("region must intersect image bounds"), so both pipelines run full width.
# Run with a long nbconvert timeout / on a capable machine.
DEM = 'Copernicus 30m Global DEM'; RESAMPLING = 'BISINC_5_POINT_INTERPOLATION'
COH_WIN_M = '100'; ORBIT = 'Sentinel Precise (Auto Download)'
print('reference:', reference_slc); print('secondary:', secondary_slc)

---
##### ***Classical InSAR pipeline (radar-geometry coregistration, geocoded at the end)***

In [ ]:
def build_iw_branch(g, slc_path, tag):
    g.add_node(operator=Operator('Read', file=slc_path), node_id=f'Read_{tag}')
    g.add_node(operator=Operator('Apply-Orbit-File', orbitType=ORBIT, continueOnFail='true'),
               node_id=f'Orbit_{tag}', source=f'Read_{tag}')
    g.add_node(operator=Operator('TOPSAR-Split', subswath=SUBSWATH, selectedPolarisations=POL,
                                 wktAoi=ETNA_WKT),
               node_id=f'Split_{tag}', source=f'Orbit_{tag}')
    return f'Split_{tag}'

gc = Graph()
r = build_iw_branch(gc, reference_slc, 'ref'); s = build_iw_branch(gc, secondary_slc, 'sec')
gc.add_node(operator=Operator('Back-Geocoding', demName=DEM, resamplingType=RESAMPLING),
            node_id='BackGeo', source=[r, s])
# Single burst -> Back-Geocoding only (ESD needs >=2 bursts; it operates on burst overlaps).
gc.add_node(operator=Operator('Interferogram', subtractFlatEarthPhase='true',
                              subtractTopographicPhase='false', includeCoherence='true',
                              cohWinSizeMeters=COH_WIN_M, cohWinAz='10', cohWinRg='10'),
            node_id='Ifg', source='BackGeo')
gc.add_node(operator=Operator('TOPSAR-Deburst'), node_id='Deburst', source='Ifg')
gc.add_node(operator=Operator('TopoPhaseRemoval', demName=DEM), node_id='Topo', source='Deburst')
gc.add_node(operator=Operator('GoldsteinPhaseFiltering', alpha='1.0'), node_id='Gold', source='Topo')
# Geocode the wrapped interferogram so it shares the GSLC's map geometry. outputComplex=true keeps
# i_ifg/q_ifg (Range-Doppler TC drops complex bands by default, leaving only intensity+coherence);
# geocoding i/q and reconstructing phase = atan2(q,i) avoids interpolating across fringe wraps.
gc.add_node(operator=Operator('Terrain-Correction', demName=DEM, imgResamplingMethod=RESAMPLING,
                              outputComplex='true'),
            node_id='TC', source='Gold')
classical_out = os.path.join(results_dir, 'cmp_classical_insar.dim')
gc.add_node(operator=Operator('Write', file=classical_out, formatName='BEAM-DIMAP'),
            node_id='Write', source='TC')
run_graph(gc, os.path.join(graphs_dir, 'cmp_classical_insar.xml'))

In [ ]:
p_classical = ProductIO.readProduct(classical_out)
print('classical bands:', [b.getName() for b in p_classical.getBands()])

---
##### ***GSLC InSAR pipeline (geocode master, CreateStack auto-bias the slave)***

In [ ]:
def gslc_insar_pair(ref_slc, sec_slc, tag):
    """Geocode ONLY the master (from a written, orbit-applied SLC so GSLC stamps its source path),
    then hand CreateStack the master GSLC + the RAW slave SLC. CreateStack reloads the master SLC
    from the stamp, cross-correlates it against the slave to estimate the (range, azimuth) bias,
    and re-geocodes the slave with that bias before stacking."""
    gm = Graph()
    gm.add_node(operator=Operator('Read', file=ref_slc), node_id='Read')
    gm.add_node(operator=Operator('Apply-Orbit-File', orbitType=ORBIT, continueOnFail='true'),
                node_id='Orbit', source='Read')
    gm.add_node(operator=Operator('TOPSAR-Split', subswath=SUBSWATH, selectedPolarisations=POL,
                                  wktAoi=ETNA_WKT),
                node_id='Split', source='Orbit')
    master_prep = os.path.join(results_dir, f'gslc_{tag}_master_prep.dim')
    gm.add_node(operator=Operator('Write', file=master_prep, formatName='BEAM-DIMAP'),
                node_id='Write', source='Split')
    run_graph(gm, os.path.join(graphs_dir, f'gslc_{tag}_master_prep.xml'))

    g = Graph()
    g.add_node(operator=Operator('Read', file=master_prep), node_id='ReadM')
    g.add_node(operator=Operator('GSLC-Terrain-Correction', demName=DEM,
                                 imgResamplingMethod=RESAMPLING, outputFlattened='false'),
               node_id='GSLC_M', source='ReadM')
    g.add_node(operator=Operator('Read', file=sec_slc), node_id='ReadS')
    g.add_node(operator=Operator('Apply-Orbit-File', orbitType=ORBIT, continueOnFail='true'),
               node_id='OrbitS', source='ReadS')
    g.add_node(operator=Operator('TOPSAR-Split', subswath=SUBSWATH, selectedPolarisations=POL,
                                 wktAoi=ETNA_WKT),
               node_id='SplitS', source='OrbitS')
    g.add_node(operator=Operator('CreateStack', extent='Master'), node_id='Stack',
               source=['GSLC_M', 'SplitS'])
    g.add_node(operator=Operator('Interferogram', includeCoherence='true',
                                 cohWinSizeMeters=COH_WIN_M, cohWinAz='10', cohWinRg='10'),
               node_id='Ifg', source='Stack')
    g.add_node(operator=Operator('GoldsteinPhaseFiltering', alpha='1.0'), node_id='Gold', source='Ifg')
    out = os.path.join(results_dir, f'cmp_gslc_{tag}_insar.dim')
    g.add_node(operator=Operator('Write', file=out, formatName='BEAM-DIMAP'), node_id='Write', source='Gold')
    run_graph(g, os.path.join(graphs_dir, f'cmp_gslc_{tag}_insar.xml'))
    return out

gslc_out = gslc_insar_pair(reference_slc, secondary_slc, 'etna')

In [ ]:
p_gslc = ProductIO.readProduct(gslc_out)
print('GSLC bands:', [b.getName() for b in p_gslc.getBands()])

---
##### ***Comparison: coherence distribution, wrapped phase, coherence maps***

In [ ]:
plot_coherence_hist(p_classical, p_gslc, 'classical', 'GSLC')

In [ ]:
plot_phase_panels(p_classical, p_gslc, 'classical', 'GSLC')

In [ ]:
plot_coherence_panels(p_classical, p_gslc, 'classical', 'GSLC')

---
##### ***Interpretation***
Both pipelines are geocoded to the same map geometry, so their coherence histograms and wrapped-phase
panels are directly comparable.

On this 12-day Etna IW1 pair the **classical** pipeline (DEM-assisted Back-Geocoding) reaches a mean
coherence of about **0.58**, while the **GSLC** pipeline (per-SLC geocoding + `CreateStack`
cross-correlation bias) reaches about **0.27**. GSLC InSAR produces genuine interferometric signal
for Sentinel-1 TOPS — visible fringes and non-noise coherence — but is currently **lower** than the
classical workflow on this pair.

The remaining gap is a coregistration gap: the classical pipeline uses DEM-assisted Back-Geocoding
(and ESD when >=2 bursts), whereas the GSLC path here relies on a scalar cross-correlation bias that
does not yet capture TOPS azimuth / per-burst refinement or an OPERA-style polynomial bias field.
Closing that gap is the natural next step; this notebook provides the side-by-side measurement to
track progress.